# Apartment price model

Ridge regression on listing features; R^2 and MAE on a held-out split. Predictions are only made inside the observed feature range.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
rng = np.random.default_rng(22)
n = 1800
df = pd.DataFrame({
    "sqm": rng.gamma(6, 12, n).round(0),
    "rooms": rng.integers(1, 6, n),
    "age_years": rng.gamma(3, 12, n).round(0),
    "distance_center_km": rng.gamma(2, 4, n).round(1),
    "floor": rng.integers(0, 12, n),
    "balcony_sqm": rng.gamma(1.2, 3, n).round(1),
    "energy_rating": rng.integers(1, 8, n),
    "renovation_score": rng.beta(2, 2, n).round(2),
    "noise_index": rng.beta(2, 3, n).round(2),
    "green_share": rng.beta(2, 2, n).round(2),
})
df["price"] = (df["sqm"] * 2100 - df["age_years"] * 800
               - df["distance_center_km"] * 3500
               + df["renovation_score"] * 40000
               + rng.normal(0, 25000, n)).round(0)

In [3]:
num_cols = ['sqm', 'rooms', 'age_years', 'distance_center_km', 'floor', 'balcony_sqm', 'energy_rating', 'renovation_score', 'noise_index', 'green_share']
X = df[num_cols]
y = df["price"]

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=22)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

In [5]:
model = Ridge(alpha=1.0)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)
r2 = r2_score(y_te, pred)
mae = mean_absolute_error(y_te, pred)
print(f"r2={r2:.3f}  mae={mae:,.0f}")

r2=0.855  mae=21,642


In [ ]:
from scipy.stats import ttest_ind
cols_to_test = ['sqm', 'rooms', 'noise_index', 'age_years', 'renovation_score', 'energy_rating', 'floor', 'distance_center_km']
significant = []
for c in cols_to_test:
    stat, p = ttest_ind(df[df['price'] > df['price'].median()][c], df[df['price'] <= df['price'].median()][c])
    if p < 0.05:
        significant.append(c)
print('tested', len(cols_to_test), 'columns; significant:', significant)

Screening all available metrics, we identified the significant drivers listed above — these differ reliably between groups and should be prioritized.

Model fit is evaluated on the held-out quarter of listings. The model describes prices within the observed range of features; we make no claims about segments outside it.